<a href="https://colab.research.google.com/github/hannaginther/ENGG680_2025_Fall/blob/Hanna/Project/GBR_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**CatBoost Gradient Boosted Regression Model for LOS Prediction**

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install catboost

In [4]:
# Import libraries
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import joblib
import json

In [5]:
# Load Data
sparcs_model_v1 = pd.read_feather('/content/drive/MyDrive/Sparcs_Datafiles/sparcs_model_v1.feather')

In [4]:
sparcs_model_v1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4238636 entries, 0 to 4238635
Data columns (total 33 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   health_service_area      object 
 1   hospital_county          object 
 2   facility_id              object 
 3   age_group                object 
 4   zip_code                 object 
 5   gender                   object 
 6   race                     object 
 7   ethnicity                object 
 8   length_of_stay           int64  
 9   admission_type           object 
 10  disposition              object 
 11  discharge_year           int64  
 12  ccsr_dx_code             object 
 13  ccsr_px_code             object 
 14  apr_drg_code             object 
 15  apr_mdc_code             object 
 16  apr_severity_code        object 
 17  apr_mortality_risk       object 
 18  apr_med_surg_desc        object 
 19  total_charges            float64
 20  payer_medicaid           int64  
 21  payer_me

In [5]:
# Prep Data
df = sparcs_model_v1.copy()

# Drop columns we don't want in model
cols_to_drop = [
    'total_charges',
    'los_log',
]
df.drop(columns=cols_to_drop, errors='ignore', inplace=True)

# In object columns, replace 'None' values with 'Missing'
obj_cols = df.select_dtypes(include=['object']).columns
df[obj_cols] = df[obj_cols].fillna('Missing')

# Save an updated dataframe
df.to_feather('/content/drive/MyDrive/Sparcs_Datafiles/sparcs_model_v2.feather')

In [6]:
model_df = pd.read_feather('/content/drive/MyDrive/Sparcs_Datafiles/sparcs_model_v2.feather')

In [8]:
# Define target + feature matrix
target_col = 'los_yj'
X = model_df.drop(columns=[target_col])
y = model_df[target_col]

# Identify categorical features
cat_features = X.select_dtypes(include=['object']).columns.tolist()
cat_features

['health_service_area',
 'hospital_county',
 'facility_id',
 'age_group',
 'zip_code',
 'gender',
 'race',
 'ethnicity',
 'admission_type',
 'disposition',
 'ccsr_dx_code',
 'ccsr_px_code',
 'apr_drg_code',
 'apr_mdc_code',
 'apr_severity_code',
 'apr_mortality_risk',
 'apr_med_surg_desc']

In [9]:
from ast import mod
# Train/Validation split
X_train_full, X_val, y_train_full, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
    )

# Downsample the training portion
train_samples = 100_000

X_train = X_train_full.sample(n=min(train_samples, len(X_train_full)), random_state=42)
y_train = y_train_full.loc[X_train.index]

print("Training size: ", X_train.shape)
print("Validation size: ", X_val.shape)

Training size:  (100000, 30)
Validation size:  (847728, 30)


In [10]:
# Create catboost pools
train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool = Pool(X_val, y_val, cat_features=cat_features)

# Define model
model = CatBoostRegressor(
    loss_function='RMSE',
    eval_metric='RMSE',
    depth=8,
    learning_rate=0.05,
    iterations=2000,
    random_seed=42,
    verbose=200,
    early_stopping_rounds=100
)

In [11]:
# Train model
model.fit(train_pool, eval_set=val_pool)

0:	learn: 0.9493843	test: 0.9528870	best: 0.9528870 (0)	total: 5.04s	remaining: 2h 47m 58s
200:	learn: 0.0022417	test: 0.0022546	best: 0.0022546 (200)	total: 8m 37s	remaining: 1h 17m 15s
400:	learn: 0.0013181	test: 0.0014470	best: 0.0014470 (400)	total: 16m 18s	remaining: 1h 5m
600:	learn: 0.0009181	test: 0.0011033	best: 0.0011033 (600)	total: 24m 33s	remaining: 57m 10s
800:	learn: 0.0007339	test: 0.0009617	best: 0.0009617 (798)	total: 33m 3s	remaining: 49m 28s
1000:	learn: 0.0006011	test: 0.0008728	best: 0.0008728 (1000)	total: 41m 28s	remaining: 41m 23s
1200:	learn: 0.0005092	test: 0.0008195	best: 0.0008195 (1200)	total: 49m 48s	remaining: 33m 8s
1400:	learn: 0.0004365	test: 0.0007805	best: 0.0007805 (1399)	total: 58m 19s	remaining: 24m 56s
1600:	learn: 0.0003798	test: 0.0007528	best: 0.0007528 (1600)	total: 1h 6m 47s	remaining: 16m 38s
1800:	learn: 0.0003340	test: 0.0007319	best: 0.0007319 (1800)	total: 1h 15m 15s	remaining: 8m 18s
1999:	learn: 0.0002980	test: 0.0007172	best: 0.0007

#**XGBoost Gradient Boosted Regression Model for LOS Prediction**